In [1]:
import numpy as np
import matplotlib.pyplot as plt
import montu as mn
import pymcel as pc

Running MontuPython version 0.10.0
Bienvenido a PyMCel v0.9.18 ¡al infinito y más allá!


1. ¿ Qué voy a hacer?: experimento para estuiar el cambio de posiicion de las estrellas debido a la aberracion. Poner a prueba la fótmula genreal de aberración, es decir la que relaciona $vec{n'}$ con $vec{n}$ 

2. ¿Qué necestio para hacerlo ? 

* calcular la velocidad de la estrella, ques es la misma de la tierra pero negativa 
* 

In [2]:
tabla , jd , X = pc.consulta_horizons(id='399' , location = '@0' , epochs = '2026-03-24 15:40:00'  )

v_tierra = X[3:]
v_tierra

array([ 1.43386170e+03, -2.98411658e+04,  1.72279316e+00])

saquemos el beta de la estrella

In [3]:
beta_vec = -v_tierra / pc.constantes.c
beta = np.linalg.norm(beta_vec)
beta_vec , beta

(array([-4.78284780e-06,  9.95394148e-05, -5.74661941e-09]),
 np.float64(9.965425611748463e-05))

In [4]:
gamma = 1 / np.sqrt(1 - beta**2)
gamma

np.float64(1.0000000049654854)

In [5]:
allstars = mn.Stars()
star = allstars.get_stars(ProperName = 'Aldebaran')
star

Loading stellar catalogue montu_stellar_catalogue_v38.csv


1 star(s):
|    |   MN |    HD |   HR |   HIP | Gl        | Name      | OtherDesignations                                                                     | ProperName   | Bayer   | Flamsteed   | Constellation   |   RAJ2000 |   DecJ2000 |   GalLonJ2000 |   GalLatJ2000 |   pmRA |   pmDec |   RadVel |   Distance |   Vmag |   Vmag_min |   Vmag_max |   B-V | SpType   |   Luminosity |   XJ2000 |   YJ2000 |   ZJ2000 |   VXJ2000 |   VYJ2000 |   VZJ2000 |   Primary | MultipleID   |   IsMultiple |   IsVariable |
|----|------|-------|------|-------|-----------|-----------|---------------------------------------------------------------------------------------|--------------|---------|-------------|-----------------|-----------|------------|---------------|---------------|--------|---------|----------|------------|--------|------------|------------|-------|----------|--------------|----------|----------|----------|-----------|-----------|-----------|-----------|--------------|--------------|---

In [7]:
ra = np.array(star.data.RAJ2000)[0]
dec = np.array(star.data.DecJ2000)[0]

mn.Util.dec2hex(ra) , mn.Util.dec2hex(dec)

('04:35:55.237', '16:30:33.484')

# Necesito obtenre $\vec{n'}$

In [8]:
import spiceypy as spy

deg = np.pi /180
rad = 1/deg

In [9]:
nprima_equ = spy.latrec(1 , ra*15*deg, dec*deg)
nprima_equ

array([0.34390374, 0.89497322, 0.28417099])

In [10]:
Requ2ecl = spy.pxform('J2000' , 'ECLIPJ2000',0)
Requ2ecl

array([[ 1.        ,  0.        ,  0.        ],
       [ 0.        ,  0.91748206,  0.39777716],
       [ 0.        , -0.39777716,  0.91748206]])

In [11]:
nprima_ecl = spy.mxv(Requ2ecl, nprima_equ)
nprima_ecl

array([ 0.34390374,  0.9341586 , -0.09527812])

In [12]:
n = nprima_ecl + ((gamma - 1)*beta_vec@nprima_ecl + gamma*beta_vec)/ (gamma*1 +  beta_vec@nprima_ecl)
n


array([ 0.34389896,  0.93425813, -0.09527812])

In [18]:
alpha = np.arccos((n @ nprima_ecl) / (np.linalg.norm(n) * np.linalg.norm(nprima_ecl)))

alpha * rad

np.float64(0.002282613780712804)

In [20]:
n_equ = spy.mxv(spy.invert(Requ2ecl),n)
n_equ

array([0.34389896, 0.89506454, 0.28421057])

# calcular la  RA y Dec despues de la aberracion

In [ ]:
r , lon , lat = spy.reclat(n_equ)

